# LeetCode 3821 — Find Nth Smallest Integer With K One Bits bằng Python

- Link: https://leetcode.com/problems/find-nth-smallest-integer-with-k-one-bits/
- Độ khó: **Hard**
- Chủ đề: Tổ hợp (đếm số có đúng `k` bit 1), xếp hạng / giải hạng tổ hợp (combinatorial unranking), Gosper's hack, tìm nhị phân trên đáp án + digit DP nhị phân

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def nthSmallest(self, n: int, k: int) -> int:
```

## 1. Mục tiêu học

1. Đếm: số các số nguyên **nhỏ hơn `2^i`** có đúng `k` bit 1 là `C(i, k)` (chọn `k` vị trí trong `i` bit thấp).
2. **Giải hạng** (unranking): dựng đáp án từ bit cao xuống, mỗi bit quyết định "đặt 1 hay 0" dựa trên so sánh `n` với `C(i, k)`.
3. Sinh lần lượt các số cùng số bit 1 bằng **Gosper's hack** (`O(1)` mỗi bước).
4. Cách khác: **tìm nhị phân** trên đáp án với hàm đếm `count(x)` = số số `≤ x` có đúng `k` bit 1.
5. Kiểm thử với duyệt trực tiếp.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `n`, `k`. Tìm số nguyên dương **nhỏ thứ `n`** có **đúng `k`** bit 1 trong biểu diễn nhị phân. Bảo đảm đáp án `< 2^{50}`.

**Ví dụ 1:** `n = 4, k = 2` → `9` (dãy `3, 5, 6, 9`).

**Ví dụ 2:** `n = 3, k = 1` → `4` (dãy `1, 2, 4`).

**Ràng buộc:** `1 ≤ n ≤ 2^{50}`, `1 ≤ k ≤ 50`.

In [1]:
from math import comb
import random, time

examples = [(4, 2, 9), (3, 1, 4)]
for n, k, exp in examples:
    print(n, k, "-> kỳ vọng", exp)

4 2 -> kỳ vọng 9
3 1 -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Đếm bit 1: `bin(x).count('1')` / `x.bit_count()`

In [2]:
for x in (3, 5, 6, 9, 255):
    print(x, bin(x), bin(x).count('1'))

3 0b11 2
5 0b101 2
6 0b110 2
9 0b1001 2
255 0b11111111 8


### 3.2. `C(i, k)` = số số trong `[0, 2^i)` có đúng `k` bit 1

Mỗi số `< 2^i` là một chuỗi `i` bit; chọn `k` vị trí đặt bit 1 ⇒ `C(i, k)` (bằng 0 nếu `k > i`). `math.comb` tính chính xác với số lớn.

In [3]:
i, k = 5, 2
print(comb(i, k), sum(1 for x in range(2**i) if bin(x).count('1') == k))
print("comb(3, 5) =", comb(3, 5))

10 10
comb(3, 5) = 0


### 3.3. Tam giác Pascal (tiền tính tổ hợp)

`C[i][j] = C[i−1][j−1] + C[i−1][j]`. Tiền tính cho `i ≤ 50` — dùng khi muốn tránh gọi `comb` nhiều lần.

In [4]:
MX = 51
C = [[0] * (MX + 1) for _ in range(MX + 1)]
for a in range(MX + 1):
    C[a][0] = 1
    for b in range(1, a + 1):
        C[a][b] = C[a - 1][b - 1] + C[a - 1][b]
print(C[50][25], comb(50, 25))

126410606437752 126410606437752


### 3.4. Gosper's hack — số tiếp theo cùng số bit 1

Với `x > 0`: `c = x & -x` (bit thấp nhất), `r = x + c` (gộp khối bit 1 thấp nhất lên), `next = (((r ^ x) >> 2) // c) | r`. Kết quả là số **nhỏ nhất lớn hơn `x`** có cùng số bit 1.

In [5]:
def gosper(x):
    c = x & -x
    r = x + c
    return (((r ^ x) >> 2) // c) | r

x = 0b11
seq = [x]
for _ in range(6):
    x = gosper(x); seq.append(x)
print(seq, [bin(v) for v in seq])

[3, 5, 6, 9, 10, 12, 17] ['0b11', '0b101', '0b110', '0b1001', '0b1010', '0b1100', '0b10001']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Liệt kê các số có `k = 2` bit 1 theo thứ tự tăng: `3 (11), 5 (101), 6 (110), 9 (1001), 10, 12, 17, …`. Nhóm theo **bit cao nhất**:

- bit cao nhất ở vị trí 1: `3` — `C(1, 1) = 1` số;
- vị trí 2: `5, 6` — `C(2, 1) = 2` số;
- vị trí 3: `9, 10, 12` — `C(3, 1) = 3` số.

Muốn số thứ `n = 4`: bỏ qua các số `< 2^3` (có `C(3, 2) = 3` số) ⇒ bit 3 phải bật, còn tìm số thứ `4 − 3 = 1` có `1` bit 1 trong 3 bit thấp ⇒ `1` ⇒ đáp án `8 + 1 = 9` ✔.

**Quy tắc tổng quát (từ bit cao xuống):** tại bit `i`, có `C(i, k)` số "đặt bit `i` = 0" (dùng `k` bit trong `i` bit thấp hơn). Nếu `n > C(i, k)` thì bit `i` **phải là 1**: trừ `n −= C(i, k)`, `k −= 1`. Ngược lại bit `i` là 0.

In [6]:
# Liệt kê 15 số đầu có k = 2 bit 1, nhóm theo vị trí bit cao nhất
from itertools import groupby
k = 2
nums = [x for x in range(1, 200) if bin(x).count('1') == k][:15]
for hb, grp in groupby(nums, key=lambda x: x.bit_length() - 1):
    g = list(grp)
    print(f"bit cao nhất = {hb}: {g}  (số lượng {len(g)} = C({hb}, {k - 1}) = {comb(hb, k - 1)})")

bit cao nhất = 1: [3]  (số lượng 1 = C(1, 1) = 1)
bit cao nhất = 2: [5, 6]  (số lượng 2 = C(2, 1) = 2)
bit cao nhất = 3: [9, 10, 12]  (số lượng 3 = C(3, 1) = 3)
bit cao nhất = 4: [17, 18, 20, 24]  (số lượng 4 = C(4, 1) = 4)
bit cao nhất = 5: [33, 34, 36, 40, 48]  (số lượng 5 = C(5, 1) = 5)


**Hai cách nhìn cùng một con số.** Nhóm "bit cao nhất ở vị trí `h`" có `C(h, k−1)` số (bit `h` cố định, chọn `k − 1` bit trong `h` bit thấp). Cộng dồn các nhóm `h = k−1, …, i−1` cho **tất cả** số `< 2^i` có `k` bit: `Σ_{h<i} C(h, k−1) = C(i, k)` — chính là **đẳng thức gậy khúc côn cầu** (hockey-stick identity). Thuật toán chỉ dùng vế phải `C(i, k)`, nên không cần cộng dồn từng nhóm.

In [7]:
for i, k in [(5, 2), (10, 3), (50, 7)]:
    print(f"i={i}, k={k}: Σ C(h, k-1) = {sum(comb(h, k - 1) for h in range(i))}, C(i, k) = {comb(i, k)}")

i=5, k=2: Σ C(h, k-1) = 10, C(i, k) = 10
i=10, k=3: Σ C(h, k-1) = 120, C(i, k) = 120
i=50, k=7: Σ C(h, k-1) = 99884400, C(i, k) = 99884400


## 5. Cách 1 — Brute Force: duyệt từng số

Tăng `x` từ 1, đếm các số có đúng `k` bit 1 tới khi đủ `n`. Thời gian tỉ lệ với **giá trị** đáp án — tới `2^{50}`, không khả thi.

In [8]:
def brute(n, k):
    x = 0
    while n:
        x += 1
        if bin(x).count('1') == k:
            n -= 1
    return x

for n, k, exp in examples:
    print(brute(n, k), "kỳ vọng", exp)

9 kỳ vọng 9
4 kỳ vọng 4


**Nút thắt:** duyệt cả những số **không** có `k` bit 1. Gosper's hack bỏ qua chúng; giải hạng tổ hợp còn bỏ qua cả việc liệt kê.

## 6. Ý tưởng thuật toán / bước đột phá

**Giải hạng tổ hợp.** Gọi `cnt(i, k) = C(i, k)` là số chuỗi `i` bit có đúng `k` bit 1 — cũng chính là số số "nhỏ hơn" khi bit `i` được đặt 0. Duyệt `i = 49, 48, …, 0`:

- nếu `n > C(i, k)`: số thứ `n` không nằm trong nhóm "bit `i` = 0" ⇒ bật bit `i`, `n −= C(i, k)`, `k −= 1`;
- ngược lại: bit `i` = 0.

Dừng khi `k = 0` (đã đặt đủ bit 1; lúc đó `n` phải bằng 1).

**Tại sao chỉ `O(50)`?** Mỗi bit xét một lần, mỗi lần một phép so sánh với tổ hợp tiền tính.

**Ví dụ minh họa quy tắc với `k = 3`, `n = 10`.** Số các số `< 2^i` có 3 bit 1: `C(3,3) = 1`, `C(4,3) = 4`, `C(5,3) = 10`, `C(6,3) = 20`.

- Bit 5: `n = 10 ≤ C(5, 3) = 10` ⇒ số thứ 10 vẫn `< 2^5` ⇒ bit 5 = 0 (nó là số **lớn nhất** dưới `2^5`).
- Bit 4: `10 > C(4, 3) = 4` ⇒ bật bit 4, `n = 6`, `k = 2`.
- Bit 3: `6 > C(3, 2) = 3` ⇒ bật bit 3, `n = 3`, `k = 1`.
- Bit 2: `3 > C(2, 1) = 2` ⇒ bật bit 2, `n = 1`, `k = 0` ⇒ dừng.

Đáp án `16 + 8 + 4 = 28 = 11100₂` — đúng là số lớn nhất có 3 bit 1 dưới `32`. Hàm trace ở mục 8 in lại đúng chuỗi quyết định này.

## 7. Các cách tiếp theo

- **Cách 2:** Gosper's hack — bắt đầu từ `2^k − 1` (số nhỏ nhất có `k` bit 1), nhảy `n − 1` bước — `O(n)`.
- **Cách 3:** giải hạng tổ hợp — `O(50)` (lời giải nộp chính).
- **Cách 4:** tìm nhị phân trên đáp án với `count(x)` (digit DP trên bit) — `O(50 · 50)`.

### 7.1. Cách 2 — Gosper's hack

In [9]:
def gosper_solution(n, k):
    x = (1 << k) - 1
    for _ in range(n - 1):
        x = gosper(x)
    return x

for n, k, exp in examples:
    print(gosper_solution(n, k), "kỳ vọng", exp)

9 kỳ vọng 9
4 kỳ vọng 4


**Độ phức tạp:** `O(n)` — tốt khi `n ≤ 10^6`, nhưng `n` tới `2^{50}` thì không được.

### 7.2. Cách 3 — Giải hạng tổ hợp (lời giải chính)

**Giải thích từng đoạn:**

1. Tiền tính tam giác Pascal `C` tới 50.
2. `ans = 0`; duyệt `i` từ 49 xuống 0.
3. Nếu `n > C[i][k]`: `n −= C[i][k]`, bật bit `i`, `k −= 1`; nếu `k == 0` thì dừng.

In [10]:
def unrank(n, k):
    ans = 0
    for i in range(49, -1, -1):
        if n > C[i][k]:
            n -= C[i][k]
            ans |= 1 << i
            k -= 1
            if k == 0:
                break
    return ans

for n, k, exp in examples:
    print(unrank(n, k), "kỳ vọng", exp)

9 kỳ vọng 9
4 kỳ vọng 4


**Độ phức tạp Cách 3:** `O(50)` mỗi truy vấn (+ `O(50²)` tiền tính một lần), bộ nhớ `O(50²)`.

### 7.3. Cách 4 — Tìm nhị phân trên đáp án

`count(x)` = số số trong `[1, x]` có đúng `k` bit 1: duyệt bit của `x` từ cao xuống; mỗi bit 1 tại vị trí `i` cho phép "đặt 0 ở đây và tự do ở `i` bit thấp" ⇒ cộng `C(i, k − đã_dùng)`. Cuối cùng cộng 1 nếu chính `x` có đúng `k` bit. Hàm `count` không giảm ⇒ tìm `x` nhỏ nhất với `count(x) ≥ n`.

In [11]:
def count_le(x, k):
    """số y trong [0, x] có đúng k bit 1 (k >= 1 nên y = 0 không được tính)."""
    res, used = 0, 0
    for i in range(x.bit_length() - 1, -1, -1):
        if x >> i & 1:
            if k - used >= 0:
                res += C[i][k - used] if k - used <= i else 0
            used += 1
            if used > k:
                return res
    return res + (1 if used == k else 0)

def binary_search_solution(n, k):
    lo, hi = 1, (1 << 50) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le(mid, k) >= n:
            hi = mid
        else:
            lo = mid + 1
    return lo

for n, k, exp in examples:
    print(binary_search_solution(n, k), "kỳ vọng", exp)
print("count_le(9, 2) =", count_le(9, 2), "(3, 5, 6, 9)")

9 kỳ vọng 9
4 kỳ vọng 4
count_le(9, 2) = 4 (3, 5, 6, 9)


In [12]:
class SolutionBinarySearch:
    def nthSmallest(self, n: int, k: int) -> int:
        return binary_search_solution(n, k)

print([SolutionBinarySearch().nthSmallest(n, k) for n, k, _ in examples])

[9, 4]


**Độ phức tạp Cách 4:** 50 bước nhị phân × `O(50)` cho `count_le` ⇒ `O(2500)`.

## 8. TRACE từng bước

Trace Cách 3 với `n = 4`, `k = 2` (chỉ in các bit có `C[i][k] > 0` hoặc có thay đổi).

In [13]:
def trace(n, k):
    ans = 0
    for i in range(49, -1, -1):
        c = C[i][k]
        if n > c:
            print(f"bit {i}: n={n} > C({i},{k})={c} -> bật bit {i}, n = {n - c}, k = {k - 1}")
            n -= c; ans |= 1 << i; k -= 1
            if k == 0:
                break
        elif c > 0 and i < 6:
            print(f"bit {i}: n={n} <= C({i},{k})={c} -> bit {i} = 0")
    print("đáp án:", ans, bin(ans))
    return ans

trace(4, 2)

bit 5: n=4 <= C(5,2)=10 -> bit 5 = 0
bit 4: n=4 <= C(4,2)=6 -> bit 4 = 0
bit 3: n=4 > C(3,2)=3 -> bật bit 3, n = 1, k = 1
bit 2: n=1 <= C(2,1)=2 -> bit 2 = 0
bit 1: n=1 <= C(1,1)=1 -> bit 1 = 0
bit 0: n=1 > C(0,1)=0 -> bật bit 0, n = 1, k = 0
đáp án: 9 0b1001


9

In [14]:
trace(10, 3)
print("vét cạn:", brute(10, 3))

bit 5: n=10 <= C(5,3)=10 -> bit 5 = 0
bit 4: n=10 > C(4,3)=4 -> bật bit 4, n = 6, k = 2
bit 3: n=6 > C(3,2)=3 -> bật bit 3, n = 3, k = 1
bit 2: n=3 > C(2,1)=2 -> bật bit 2, n = 1, k = 0
đáp án: 28 0b11100
vét cạn: 28


## 9. Code nộp LeetCode

In [15]:
MX = 50
COMB = [[0] * (MX + 1) for _ in range(MX + 1)]
for a in range(MX + 1):
    COMB[a][0] = 1
    for b in range(1, a + 1):
        COMB[a][b] = COMB[a - 1][b - 1] + COMB[a - 1][b]

class Solution:
    def nthSmallest(self, n: int, k: int) -> int:
        ans = 0
        for i in range(49, -1, -1):
            c = COMB[i][k]                 # số chuỗi i bit có đúng k bit 1 (bit i = 0)
            if n > c:
                n -= c
                ans |= 1 << i
                k -= 1
                if k == 0:
                    break
        return ans

sol = Solution()
for n, k, exp in examples:
    got = sol.nthSmallest(n, k)
    print((n, k), "->", got, "✔" if got == exp else "✘")

(4, 2) -> 9 ✔
(3, 1) -> 4 ✔


## 10. Vì sao thuật toán đúng

1. **Thứ tự số = thứ tự từ điển trên chuỗi 50 bit** (cùng độ dài, có số 0 đầu). Các số có bit `i` = 0 (với các bit cao hơn đã cố định) đều **nhỏ hơn** các số có bit `i` = 1.
2. **Bất biến:** trước khi xét bit `i`, các bit cao hơn đã cố định, còn cần đặt `k` bit 1 trong `i + 1` bit thấp, và đáp án là số **thứ `n`** (theo thứ tự tăng) trong nhóm đó. Nhóm "bit `i` = 0" có `C(i, k)` phần tử, đứng trước nhóm "bit `i` = 1". Nếu `n ≤ C(i, k)` thì đáp án ở nhóm đầu (bit = 0, giữ `n`, `k`); ngược lại ở nhóm sau với thứ hạng `n − C(i, k)` và còn `k − 1` bit.
3. **Dừng:** khi `k = 0`, nhóm còn lại chỉ có một phần tử (mọi bit thấp bằng 0) nên `n = 1`; các bit thấp đều 0.
4. **Tồn tại:** đề bảo đảm đáp án `< 2^{50}` ⇒ `n ≤ C(50, k)`, nên quá trình luôn hoàn tất trong 50 bit.

**Liên hệ với hệ đếm tổ hợp (combinatorial number system).** Mọi số nguyên `N ≥ 0` biểu diễn **duy nhất** dưới dạng `N = C(c_k, k) + C(c_{k−1}, k−1) + … + C(c_1, 1)` với `c_k > c_{k−1} > … > c_1 ≥ 0`. Các vị trí `c_j` chính là các bit 1 của số có hạng `N + 1` trong dãy các số có `k` bit 1. Thuật toán giải hạng thực chất là tìm biểu diễn này một cách tham lam: mỗi lần chọn `c` lớn nhất với `C(c, k) < n`.

In [16]:
def combinadic(N, k):
    # tham lam: c_k lớn nhất với C(c_k, k) <= N
    cs = []
    for j in range(k, 0, -1):
        c = j - 1
        while comb(c + 1, j) <= N:
            c += 1
        cs.append(c); N -= comb(c, j)
    return cs

for n, k in [(4, 2), (10, 3), (123456, 5)]:
    cs = combinadic(n - 1, k)
    print((n, k), "-> vị trí bit:", cs, "-> số", sum(1 << c for c in cs), "| lời giải:", Solution().nthSmallest(n, k))

(4, 2) -> vị trí bit: [3, 0] -> số 9 | lời giải: 9
(10, 3) -> vị trí bit: [4, 3, 2] -> số 28 | lời giải: 28
(123456, 5) -> vị trí bit: [29, 19, 18, 4, 2] -> số 537657364 | lời giải: 537657364


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt từng số | `O(đáp án)` ≤ `2^{50}` | `O(1)` |
| 2. Gosper's hack | `O(n)` | `O(1)` |
| 3. Giải hạng tổ hợp | `O(50)` (+ `O(50²)` tiền tính) | `O(50²)` |
| 4. Nhị phân + đếm | `O(50 · 50)` | `O(50²)` |

## 12. Bộ test

In [17]:
def run_all(n, k):
    res = [unrank(n, k), binary_search_solution(n, k), Solution().nthSmallest(n, k)]
    if n <= 10**5:
        res.append(gosper_solution(n, k))
    if res[0] < 2**16:                       # vét cạn chỉ khi đáp án nhỏ
        res.append(brute(n, k))
    return res

for n, k, exp in examples:
    assert all(v == exp for v in run_all(n, k))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [18]:
# 12.2 ca biên (tự dựng)
cases = [
    (1, 1, 1), (1, 5, 31), (1, 50, 2**50 - 1),          # số nhỏ nhất: 2^k - 1
    (50, 1, 2**49),                                     # k = 1: lũy thừa của 2
    (comb(50, 25), 25, None),                           # số lớn nhất có 25 bit trong 50 bit
    (comb(50, 2), 2, 2**49 + 2**48),
]
for n, k, exp in cases:
    r = run_all(n, k)
    assert len(set(r)) == 1, (n, k, r)
    if exp is not None:
        assert r[0] == exp
    print((n, k), "->", r[0], bin(r[0])[:20] + ("..." if r[0] > 2**18 else ""))

(1, 1) -> 1 0b1
(1, 5) -> 31 0b11111
(1, 50) -> 1125899906842623 0b111111111111111111...
(50, 1) -> 562949953421312 0b100000000000000000...
(126410606437752, 25) -> 1125899873288192 0b111111111111111111...
(1225, 2) -> 844424930131968 0b110000000000000000...


In [19]:
# 12.3 phản ví dụ cho suy đoán sai "đáp án = (2^k - 1) + (n - 1)" (coi các số liên tiếp)
def wrong_consecutive(n, k):
    return (1 << k) - 1 + (n - 1)

for n, k in [(1, 2), (2, 2), (4, 2)]:
    print((n, k), "suy đoán:", wrong_consecutive(n, k), "| đúng:", Solution().nthSmallest(n, k))

(1, 2) suy đoán: 3 | đúng: 3
(2, 2) suy đoán: 4 | đúng: 5
(4, 2) suy đoán: 6 | đúng: 9


In [20]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3821)
for _ in range(300):
    k = rng.randint(1, 8); n = rng.randint(1, min(3000, comb(50, k)))
    r = run_all(n, k)
    assert len(set(r)) == 1, (n, k, r)
for _ in range(300):
    k = rng.randint(1, 50); n = rng.randint(1, comb(50, k))
    r = run_all(n, k)
    assert len(set(r)) == 1 and bin(r[0]).count('1') == k and r[0] < 2**50, (n, k, r)
print("600 test ngẫu nhiên: OK")

600 test ngẫu nhiên: OK


In [21]:
# 12.5 tính chất: hạng (rank) của đáp án đúng bằng n — kiểm tra bằng count_le
rng = random.Random(9)
ok = True
for _ in range(500):
    k = rng.randint(1, 50); n = rng.randint(1, comb(50, k))
    x = Solution().nthSmallest(n, k)
    ok &= count_le(x, k) == n and count_le(x - 1, k) == n - 1
print("count_le(x) = n và count_le(x-1) = n-1 với mọi test:", ok)
t0 = time.time()
for _ in range(10**4): Solution().nthSmallest(rng.randint(1, comb(50, 25)), 25)
print("10^4 truy vấn:", round(time.time() - t0, 3), "s")

count_le(x) = n và count_le(x-1) = n-1 với mọi test: True
10^4 truy vấn: 0.057 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh `n ≥ C(i, k)` thay vì `n > C(i, k)`

Nhóm "bit `i` = 0" chứa `C(i, k)` số với hạng `1..C(i, k)`. Nếu `n = C(i, k)` thì đáp án vẫn ở nhóm này (số cuối cùng của nhóm). Dùng `≥` làm lệch một.

In [22]:
def wrong_ge_simple(n, k):
    ans = 0
    for i in range(49, -1, -1):
        if n >= C[i][k] and k > 0:          # SAI: phải là >
            n -= C[i][k]; ans |= 1 << i; k -= 1
    return ans

for n, k in [(3, 2), (4, 2), (1, 1)]:
    print((n, k), "sai:", wrong_ge_simple(n, k), "| đúng:", Solution().nthSmallest(n, k))

(3, 2) sai: 9 | đúng: 6
(4, 2) sai: 10 | đúng: 9
(1, 1) sai: 2 | đúng: 1


### Lỗi 2 — Bắt đầu từ bit quá thấp (ví dụ bit 31)

Đáp án có thể tới `2^{50} − 1`; bắt đầu từ bit 31 sẽ bỏ sót bit cao và cho kết quả sai (hoặc vòng lặp kết thúc khi `k` chưa về 0).

In [23]:
def wrong_low_start(n, k):
    ans = 0
    for i in range(31, -1, -1):                 # SAI: phải từ bit 49
        if n > C[i][k]:
            n -= C[i][k]; ans |= 1 << i; k -= 1
            if k == 0: break
    return ans

n, k = comb(40, 3) + 5, 3
print("sai:", wrong_low_start(n, k), "| đúng:", Solution().nthSmallest(n, k))

sai: 3758096384 | đúng: 1099511627786


### Lỗi 3 — Dùng `float` cho tổ hợp lớn

`C(50, 25) ≈ 1,26·10^{14}` vẫn vừa `float`, nhưng các phép trừ với `n` tới `2^{50}` có thể mất chính xác nếu dùng công thức giai thừa dạng thực (`math.factorial` chia `/`). Luôn dùng số nguyên (`math.comb` hoặc Pascal).

In [24]:
from math import factorial
approx = factorial(50) / (factorial(25) * factorial(25))
print("float:", approx, "| nguyên:", comb(50, 25), "| bằng nhau?", approx == comb(50, 25))
big = factorial(60) / (factorial(30) * factorial(30))
print("C(60,30) float == nguyên?", big == comb(60, 30))

float: 126410606437752.0 | nguyên: 126410606437752 | bằng nhau? True
C(60,30) float == nguyên? True


### Lỗi 4 — Gosper's hack bắt đầu từ `1` thay vì `2^k − 1`

Gosper giữ nguyên số bit 1; bắt đầu từ `1` chỉ sinh các số có **một** bit.

In [25]:
x = 1
print("bắt đầu từ 1:", [x := gosper(x) for _ in range(4)], "| đúng (k=2) phải là 5, 6, 9, 10")

bắt đầu từ 1: [2, 4, 8, 16] | đúng (k=2) phải là 5, 6, 9, 10


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Số nhỏ thứ `n` có tính chất X"** mà đếm được số phần tử `< 2^i` thỏa X ⇒ **giải hạng**: dựng từ bit cao xuống, so `n` với số lượng ở nhánh 0.
- Hoặc **tìm nhị phân trên đáp án** + hàm đếm `count(x)` (digit DP).
- **Cùng số bit 1, liệt kê theo thứ tự** ⇒ Gosper's hack.
- Liên quan: LC 60 *Permutation Sequence* (giải hạng hoán vị), LC 440 *K-th Smallest in Lexicographical Order*.

- **Kiểm tra nhanh bằng cận:** đáp án luôn nằm trong `[2^k − 1, 2^{50})`; số lớn nhất có `k` bit 1 dưới `2^{50}` là `(2^k − 1) << (50 − k)`, có hạng `C(50, k)`. Hai "điểm neo" `n = 1` và `n = C(50, k)` là ca biên tốt để thử.
- **Đừng duyệt:** ngay cả với `k = 25`, mật độ các số có 25 bit 1 trong `[0, 2^{50})` chỉ khoảng `C(50,25)/2^{50} ≈ 11%`, nhưng `2^{50}` vẫn quá lớn để duyệt.

In [26]:
for k in (1, 10, 25, 50):
    top = ((1 << k) - 1) << (50 - k)
    print(f"k={k}: n=C(50,k) -> {Solution().nthSmallest(comb(50, k), k) == top}, mật độ ≈ {comb(50, k) / 2**50:.2%}")

k=1: n=C(50,k) -> True, mật độ ≈ 0.00%
k=10: n=C(50,k) -> True, mật độ ≈ 0.00%
k=25: n=C(50,k) -> True, mật độ ≈ 11.23%
k=50: n=C(50,k) -> True, mật độ ≈ 0.00%


## 15. Biến thể của bài

1. **Ngược lại — xếp hạng (rank):** cho `x` có `k` bit 1, tìm thứ tự của nó ⇒ `count_le(x, k)`.
2. **Số thứ `n` có số bit 1 trong `[a, b]`**: thay `C(i, k)` bằng `Σ_{j} C(i, j)`.
3. **Cơ số khác** (số thứ `n` có đúng `k` chữ số khác 0 trong hệ `b`): `C(i, k)·(b−1)^k`.
4. **Không có số 0 kề nhau / không có hai bit 1 kề nhau**: thay `C` bằng DP đếm (Fibonacci).

In [27]:
# Biến thể 4: số thứ n (n >= 1) KHÔNG có hai bit 1 kề nhau (Zeckendorf-like), số dương
def nth_no_adjacent(n):
    # F[i] = số chuỗi i bit không có hai 1 kề nhau (kể cả chuỗi toàn 0)
    F = [1, 2]
    for _ in range(60): F.append(F[-1] + F[-2])
    n += 1                                # tính cả số 0 ở hạng 1
    ans = 0
    for i in range(59, -1, -1):
        if n > F[i]:
            n -= F[i]; ans |= 1 << i
    return ans

seq = [x for x in range(1, 100) if x & (x >> 1) == 0][:12]
print([nth_no_adjacent(i) for i in range(1, 13)], seq)

[1, 2, 4, 5, 8, 9, 10, 16, 17, 18, 20, 21] [1, 2, 4, 5, 8, 9, 10, 16, 17, 18, 20, 21]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt | Đếm từng số | `O(đáp án)` | `O(1)` | Kiểm thử |
| 2. Gosper | Nhảy tới số cùng popcount kế tiếp | `O(n)` | `O(1)` | `n` nhỏ, cần liệt kê |
| 3. Giải hạng | So `n` với `C(i, k)` từ bit cao | `O(50)` | `O(50²)` | **Lời giải chuẩn** |
| 4. Nhị phân + đếm | `count(x) ≥ n` | `O(50²)` | `O(50²)` | Khi chỉ có hàm đếm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `nthSmallest(5, 2)` = ?

**Bài 2 (tính tay).** `nthSmallest(7, 3)` bằng giải hạng.

**Bài 3 (điền TODO).** Viết `rank(x)` = thứ hạng của `x` trong dãy các số có cùng số bit 1 với `x`.

**Bài 4 (tự viết lại).** Viết lời giải đệ quy: `f(n, k, i)` trả về đáp án dùng các bit `0..i`.

**Bài 5 (mini challenge).** Số thứ `n` có số bit 1 **chẵn** (dãy "evil numbers", tính cả 0 hay không — tự quy ước và kiểm chứng).

In [28]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def rank(x):
    # TODO: count_le(x, popcount(x))
    pass

In [29]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def nth_even_popcount(n):
    # TODO: số chuỗi i bit có số bit 1 chẵn = 2^(i-1) (i >= 1); giải hạng theo tính chẵn lẻ còn lại
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dãy `3, 5, 6, 9, 10`. (2) Thứ 5 là `10`. (3) Kiểm tra bằng giải hạng: `n = 5 > C(3,2) = 3` ⇒ bit 3, còn `n = 2`, `k = 1`; `2 > C(1,1) = 1` ⇒ bit 1 ⇒ `8 + 2 = 10`.

**Bài 2.** (1) `C(3,3) = 1`, `C(4,3) = 4`, `C(5,3) = 10`. (2) `7 > 4` ⇒ bit 4, `n = 3`, `k = 2`; `3 ≤ C(3,2) = 3` ⇒ bit 3 = 0; `3 > C(2,2) = 1` ⇒ bit 2, `n = 2`, `k = 1`; `2 > C(1,1) = 1` ⇒ bit 1, `n = 1`, `k = 0`. (3) `16 + 4 + 2 = 22`.

**Bài 3.** (1) `k = bin(x).count('1')`. (2) `count_le(x, k)`. (3) `rank(nthSmallest(n, k)) = n`.

**Bài 4.** (1) Cơ sở `k = 0` ⇒ 0. (2) Nếu `n > C(i, k)` thì `2^i + f(n − C(i,k), k−1, i−1)`. (3) Ngược lại `f(n, k, i−1)`.

**Bài 5.** (1) Trạng thái: số bit còn lại và tính chẵn lẻ cần đạt. (2) Số chuỗi `i` bit có parity `p` là `2^{i−1}` (với `i ≥ 1`). (3) Đối chiếu duyệt.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1, 2
assert Solution().nthSmallest(5, 2) == 10 == brute(5, 2)
assert Solution().nthSmallest(7, 3) == 22 == brute(7, 3)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [31]:
# Bài 3
def rank_sol(x):
    return count_le(x, bin(x).count('1'))

rng = random.Random(3)
for _ in range(300):
    k = rng.randint(1, 50); n = rng.randint(1, comb(50, k))
    assert rank_sol(Solution().nthSmallest(n, k)) == n
print("Bài 3 OK")

Bài 3 OK


In [32]:
# Bài 4
def rec(n, k, i=49):
    if k == 0:
        return 0
    if n > C[i][k]:
        return (1 << i) + rec(n - C[i][k], k - 1, i - 1)
    return rec(n, k, i - 1)

rng = random.Random(4)
for _ in range(300):
    k = rng.randint(1, 50); n = rng.randint(1, comb(50, k))
    assert rec(n, k) == Solution().nthSmallest(n, k)
print("Bài 4 OK")

Bài 4 OK


In [33]:
# Bài 5: số dương thứ n có số bit 1 chẵn (không tính 0)
def nth_even_popcount_sol(n):
    n += 1                                   # tính cả 0 (popcount 0 chẵn) ở hạng 1
    ans, parity = 0, 0                       # parity = số bit 1 đã đặt (mod 2)
    for i in range(59, -1, -1):
        cnt0 = (1 << (i - 1)) if i >= 1 else (1 if parity == 0 else 0)   # số chuỗi i bit cho tổng chẵn khi bit i = 0
        if n > cnt0:
            n -= cnt0; ans |= 1 << i; parity ^= 1
    return ans

expected = [x for x in range(1, 200) if bin(x).count('1') % 2 == 0][:30]
assert [nth_even_popcount_sol(i) for i in range(1, 31)] == expected
print("Bài 5 OK:", expected[:10])

Bài 5 OK: [3, 5, 6, 9, 10, 12, 15, 17, 18, 20]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# tiền tính C[i][j] (Pascal) tới 50
ans = 0
for i in range(49, -1, -1):
    if n > C[i][k]:          # nhóm "bit i = 0" có C[i][k] số
        n -= C[i][k]; ans |= 1 << i; k -= 1
        if k == 0: break
return ans
```

**Checklist:**

- [ ] So sánh **`n > C(i, k)`** (không phải `≥`)?
- [ ] Duyệt từ bit **49** (đáp án `< 2^{50}`)?
- [ ] Tổ hợp tính bằng số nguyên (Pascal / `math.comb`)?
- [ ] Dừng khi `k = 0`?
- [ ] Đã kiểm chứng với Gosper/duyệt cho `n` nhỏ?